# Extração de Características de Imagem + Visualização t-SNE

## Informações gerais
### Título de Projeto de Iniciação Científica FAPESP
Classificação e agrupamento de fontes tipográficas de acervo do Museu Paulista com uso de técnicas de aprendizado de máquina (Processo 2024/23738-0)
### Beneficiário
Matheus Hencklein Ponte (FEEC/UNICAMP)
### Responsável
Profa. Dra. Paula Dornhofer Paro Costa (FEEC/UNICAMP)
### Orientadoras
Orientadora principal: Profa. Dra. Paula Dornhofer Paro Costa (FEEC/UNICAMP) <br>
Coorientadora: Profa. Dra. Solange Ferraz de Lima (MP/USP)

## 0. Imports de bibliotecas Python

Bibliotecas utilizadas: <br>
<ul>
    <li>PyTorch: implementa o modelo de rede neural pré-treinado utilizado para extração de características das imagens, ResNet18;</li>
    <li>Python Image Library (PIL): permite a leitura de uma imagem como uma classe Image, representada por uma matriz;</li>
    <li>Glob: permite o uso de expressões regulares (regex) na leitura de caminhos para arquivos (pathnames), que seguem um determinado padrão;</li>
    <li>Numpy: implementa a estrutura-padrão de um tensor, que é compartilhada entre as bibliotecas.</li>
    <li>Scikit-learn: contém a implementação do t-SNE, algoritmo de redução de dimensionalidade para visualizar os dados de alta dimensão (camadas mais profundas da rede neural) em baixa dimensão (gráfico 2D).</li>
    <li>Plotly: constrói e exibe o gráfico dos dados obtidos do algoritmo t-SNE.</li>
</ul>

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import glob
import numpy as np
from sklearn.manifold import TSNE
import os
from bokeh.plotting import figure, show
from bokeh.layouts import row
from bokeh.models import ColumnDataSource, HoverTool, CustomJS
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10
from bokeh.io import output_notebook

In [15]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

ROOT_DIRECTORY = 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed'

Mounted at /content/data


## 1. Carregando a rede neural ResNet pré-treinada como um Extrator de Características

### O modelo ResNet18
O modelo ResNet18 (18-layer Residual Network) é um modelo de Rede Neural Convolucional (CNN) residual com 18 camadas de pesos treináveis. Seu uso é particularmente interessante por preservar características de baixo nível (bordas, curvas e texturas) e de estruturas alto nível (formas e objetos), prevenindo overfitting. [https://arxiv.org/pdf/1512.03385]

### Extrator de características
O extrator de características

In [16]:
resnet = models.resnet18(pretrained=True)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])  # remove final FC
feature_extractor.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

## 2 - Preprocessing Pipeline

In [17]:
transform = transforms.Compose([
    transforms.Resize((224,224)),       # keep detail but manageable size
    transforms.Grayscale(num_output_channels=3),  # convert 1->3 channel
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## 3 - Load dataset and extract features

In [20]:
features = []
labels = []
img_paths = []

path_pattern = ROOT_DIRECTORY + '/*/*/*.png'
for img_path in glob.glob(path_pattern):
    label = img_path.split("/")[-3]   # class = folder name
    print(img_path+"\n")
    img = Image.open(img_path).convert("RGB")
    img_t = transform(img).unsqueeze(0)  # shape [1, 3, 224, 224]
    
    with torch.no_grad():
        feat = feature_extractor(img_t)   # [1, 512, 1, 1]
        feat = feat.view(-1).numpy()      # flatten -> [512]
    
    features.append(feat)
    labels.append(label)
    img_paths.append(img_path)

X = np.array(features)   # shape (n_samples, 512)
y = np.array(labels)

print("Feature matrix shape:", X.shape)
print(labels)

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_006.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_014.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_004.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_003.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_002.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_011.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_009.png

data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_016.png

data/Shareddrives/ai4museums/projetos/ti

## 4 - Running t-SNE

In [21]:
tsne = TSNE(
    n_components=2, 
    perplexity=15,   # since dataset is small
    learning_rate=200, 
    max_iter=2000,
    init="pca", 
    random_state=42
)
X_embedded = tsne.fit_transform(X)

## 5 - Plot t-SNE

In [22]:
# Show inline in Jupyter
output_notebook()

# Prepare data
classes = list(set(labels))  # unique class names
filenames = [os.path.basename(p) for p in img_paths]
print(img_paths)

source = ColumnDataSource(data=dict(
    x=X_embedded[:, 0],
    y=X_embedded[:, 1],
    label=labels,
    filename=filenames,
    path=img_paths,
))

# Left: scatter with colored points per class
scatter = figure(
    title="t-SNE of Font Samples (click to preview)",
    width=700, height=700,
    tools="pan,wheel_zoom,box_zoom,reset,tap"
)

scatter.circle(
    "x", "y", source=source, size=8, alpha=0.8,
    color=factor_cmap("label", palette=Category10[len(classes)], factors=classes),
    legend_field="label"
)

scatter.add_tools(HoverTool(tooltips=[("file", "@filename"), ("class", "@label")]))

# Style legend
scatter.legend.title = "Font Family"
scatter.legend.location = "top_left"
scatter.legend.label_text_font_size = "10pt"

# Right: preview panel
preview = figure(
    title="Preview",
    width=300, height=300,
    x_range=(0, 1), y_range=(0, 1),
    tools=""
)
preview.axis.visible = False
preview.grid.visible = False

# Image source (empty at start)
img_source = ColumnDataSource(data=dict(url=[]))
preview.image_url(url="url", x=0, y=1, w=1, h=1, anchor="top_left", source=img_source)

# JS callback to update preview when a point is clicked
callback = CustomJS(args=dict(source=source, img_source=img_source), code="""
    var indices = source.selected.indices;
    if (indices.length > 0) {
        var idx = indices[0];
        var path = source.data['path'][idx];
        img_source.data = {url: [path]};
        img_source.change.emit();
    }
""")
source.selected.js_on_change("indices", callback)

# Layout
layout = row(scatter, preview)
show(layout)

['data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_006.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_014.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_004.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_003.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_002.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_011.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_009.png', 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista/evaluation/processed/escritural/cod_135/cod_135_016.png', 'data/Shareddrives/ai4m